In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fintech_finpay.observability;

In [0]:
%sql
CREATE OR REPLACE TABLE fintech_finpay.observability.pipeline_event_log AS
SELECT *
FROM event_log('bc65d1fc-8806-48ac-99bd-14947fbbbbc5');

In [0]:
%sql
CREATE OR REPLACE VIEW fintech_finpay.observability.vw_records_by_layer AS
SELECT
    CASE
        WHEN origin.flow_name LIKE '%bronze.transactions'
          OR origin.flow_name LIKE '%bronze.merchants'
          OR origin.flow_name LIKE '%bronze.users'
          OR origin.flow_name IN ('transactions', 'merchants', 'users')
        THEN 'Bronze'

        WHEN origin.flow_name LIKE '%silver_transactions'
          OR origin.flow_name LIKE '%silver_merchants'
          OR origin.flow_name LIKE '%silver_users'
          OR origin.flow_name LIKE '%quarantine%'
          OR origin.flow_name IN ('silver_transactions', 'silver_merchants', 'silver_users', 'quarantine')
        THEN 'Silver'

        WHEN origin.flow_name LIKE '%gold_%'
        THEN 'Gold'

        ELSE 'Other'
    END AS layer,

    origin.flow_name AS dataset_name,
    DATE(timestamp) AS event_date,
    SUM(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT)) AS processed_records

FROM fintech_finpay.observability.pipeline_event_log
WHERE event_type = 'flow_progress'
GROUP BY
    CASE
        WHEN origin.flow_name LIKE '%bronze.transactions'
          OR origin.flow_name LIKE '%bronze.merchants'
          OR origin.flow_name LIKE '%bronze.users'
          OR origin.flow_name IN ('transactions', 'merchants', 'users')
        THEN 'Bronze'

        WHEN origin.flow_name LIKE '%silver_transactions'
          OR origin.flow_name LIKE '%silver_merchants'
          OR origin.flow_name LIKE '%silver_users'
          OR origin.flow_name LIKE '%quarantine%'
          OR origin.flow_name IN ('silver_transactions', 'silver_merchants', 'silver_users', 'quarantine')
        THEN 'Silver'

        WHEN origin.flow_name LIKE '%gold_%'
        THEN 'Gold'

        ELSE 'Other'
    END,
    origin.flow_name,
    DATE(timestamp);

In [0]:
%sql
CREATE OR REPLACE VIEW fintech_finpay.observability.vw_quarantine_rejections AS
SELECT
    source_name,
    rejection_reason,
    DATE(processing_timestamp) AS rejection_date,
    COUNT(*) AS rejected_records
FROM fintech_finpay.silver.quarantine
GROUP BY
    source_name,
    rejection_reason,
    DATE(processing_timestamp);

In [0]:
%sql
CREATE OR REPLACE VIEW fintech_finpay.observability.vw_quality_expectations AS
SELECT
    timestamp,
    origin.flow_name AS dataset_name,
    details:flow_progress:data_quality:expectations AS expectations
FROM fintech_finpay.observability.pipeline_event_log
WHERE event_type = 'flow_progress'
  AND details:flow_progress:data_quality:expectations IS NOT NULL;

In [0]:
%sql
SELECT *
FROM fintech_finpay.observability.vw_records_by_layer


In [0]:
%sql
SELECT *
FROM fintech_finpay.observability.vw_quarantine_rejections


In [0]:
%sql
SELECT *
FROM fintech_finpay.gold.fact_transactions